# 08 — Modèle d'arbres, arbitrage des seuils et usagers sans âge

Ce notebook prolonge les notebooks 06 et 07 sur trois points laissés ouverts :

1. **Un modèle non linéaire.** Les notebooks 06 et 07 comparent une référence constante
   à une régression logistique. La question « une méthode plus souple ferait-elle mieux ? »
   restait sans réponse chiffrée. On compare ici un modèle d'arbres boostés, avec
   exactement les mêmes variables, le même découpage et la même règle de sélection.
2. **L'arbitrage précision-rappel.** Le seuil illustratif de 0,5 donne un rappel de 18 %.
   Ce chiffre ne dit rien de ce que coûterait un rappel plus élevé. On construit ici la
   table des seuils, en les fixant sur 2023 et en les appliquant tels quels à 2024.
3. **Les usagers sans âge.** Le notebook 07 signale que ce groupe est presque exempt de
   cas graves. On date ici l'apparition du phénomène, on décrit ces usagers et on mesure
   ce que leur présence apporte aux scores.

## Limite déclarée avant toute exécution

Les scores de 2024 de la régression logistique **ont déjà été consultés** le 17 octobre 2026.
2024 n'est donc plus une année vierge pour la comparaison menée ici. Deux garde-fous :

- les réglages du modèle d'arbres sont choisis **sur 2023 seule**, avec une grille écrite
  sur disque avant le premier score ;
- aucun score de 2024 n'intervient dans ce choix, et le modèle n'est pas réentraîné après
  consultation du test.

Le protocole du cahier des charges reste inchangé. Ce notebook ajoute une comparaison,
il ne révise pas rétrospectivement les critères des étapes précédentes.

## 1. Chargement et contrôle de provenance

Comme le notebook 07, on vérifie l'empreinte des données avant de s'en servir. Un écart
doit interrompre l'exécution et être examiné.

In [ ]:
import hashlib, itertools, json, os, sys, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, log_loss,
                             precision_recall_curve, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RACINE = ".."
sys.path.insert(0, os.path.join(RACINE, "src"))
import baac_ml

DONNEES = os.path.join(RACINE, "data/processed/baac_usagers_metropole_2019_2024.csv.gz")
SCHEMA = os.path.join(RACINE, "data/processed/baac_regles_et_schema.json")
CONFIG_06 = os.path.join(RACINE, "reports/tables/06_modeling/06_configuration_retenue.json")

TABLES = os.path.join(RACINE, "reports/tables/08_comparaison")
FIGURES = os.path.join(RACINE, "reports/figures/08_comparaison")
os.makedirs(TABLES, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)


def empreinte(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as f:
        for bloc in iter(lambda: f.read(1 << 20), b""):
            h.update(bloc)
    return h.hexdigest()


config_06 = json.load(open(CONFIG_06))
schema = json.load(open(SCHEMA))

sha_donnees = empreinte(DONNEES)
assert sha_donnees == config_06["sha256_donnees"], (
    "Les données ne sont plus celles utilisées par les notebooks 06 et 07. "
    "Examiner l'écart avant de poursuivre."
)
print("Empreinte des données conforme à la configuration du notebook 06.")
print("sha256 :", sha_donnees)

In [ ]:
CIBLE = "cible_deces_hospitalisation"
APPRENTISSAGE = [2019, 2020, 2021, 2022]
VALIDATION = 2023
TEST = 2024

donnees = pd.read_csv(
    DONNEES,
    low_memory=False,
    na_values=[schema["marqueur_na_csv"]],
    dtype={"Num_Acc": "string", "id_vehicule": "string", "id_usager": "string"},
)

connus = donnees[donnees["cible_connue"] == True].copy()
apprentissage = connus[connus["annee_source"].isin(APPRENTISSAGE)].copy()
validation = connus[connus["annee_source"] == VALIDATION].copy()
test = connus[connus["annee_source"] == TEST].copy()

X_app = baac_ml.make_features(apprentissage)
X_val = baac_ml.make_features(validation)
X_tst = baac_ml.make_features(test)
y_app = apprentissage[CIBLE].astype(int).to_numpy()
y_val = validation[CIBLE].astype(int).to_numpy()
y_tst = test[CIBLE].astype(int).to_numpy()

print(f"apprentissage {len(y_app)} | validation {len(y_val)} | test {len(y_tst)}")
assert (len(y_app), len(y_val)) == (config_06["usagers_apprentissage"],
                                    config_06["usagers_validation"])

probabilite_reference = y_app.mean()
print("probabilité de référence :", probabilite_reference)
assert abs(probabilite_reference - config_06["probabilite_reference"]) < 1e-12

## 2. Garde-fou : retrouver les scores du notebook 07

Avant d'introduire quoi que ce soit de nouveau, on réentraîne la régression logistique
retenue et on vérifie qu'elle redonne bien les scores enregistrés. Sans ce contrôle,
une comparaison avec un autre modèle ne prouverait rien.

In [ ]:
reg = config_06["regression"]

logistique = Pipeline([
    ("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ("modele", LogisticRegression(
        C=config_06["C_retenu_logistique"], solver=reg["solver"], tol=reg["tol"],
        max_iter=reg["max_iter"], class_weight=reg["class_weight"],
        random_state=reg["random_state"],
    )),
])
logistique.fit(X_app, y_app)


def scores(y, p):
    return {
        "log_loss": log_loss(y, p, labels=[0, 1]),
        "brier": brier_score_loss(y, p),
        "average_precision": average_precision_score(y, p),
        "roc_auc": roc_auc_score(y, p),
    }


p_log_val = logistique.predict_proba(X_val)[:, 1]
p_log_tst = logistique.predict_proba(X_tst)[:, 1]

obtenus = scores(y_val, p_log_val)
attendus = config_06["scores_validation_logistique"]
ecart_max = max(abs(obtenus[k] - attendus[k]) for k in attendus)

reconstruction = pd.DataFrame([
    {"score": k, "obtenu": obtenus[k], "enregistre_06": attendus[k],
     "ecart": abs(obtenus[k] - attendus[k])}
    for k in attendus
])
print(reconstruction.to_string(index=False))
print(f"\nécart maximal : {ecart_max:.2e}")
assert ecart_max < 1e-4, "La reconstruction s'écarte trop : examiner l'environnement."
print("Reconstruction conforme à la tolérance du notebook 07 (1e-4).")

## 3. Protocole du modèle d'arbres, écrit avant les scores

Le fichier ci-dessous est enregistré **avant le premier entraînement**. La grille, les
paramètres fixes et la règle de départage ne sont plus modifiables ensuite.

Deux choix méritent une justification :

- **Représentation.** Les mêmes onze variables et les mêmes tranches d'âge que le
  notebook 06. Les modalités sont apprises sur 2019-2022 et déclarées en catégories
  pandas : le modèle traite nativement les variables catégorielles, sans one-hot. Une
  modalité jamais vue devient une valeur manquante, gérée comme telle, et elle est comptée.
- **Arrêt anticipé désactivé.** L'arrêt anticipé de scikit-learn découpe aléatoirement
  l'apprentissage pour se donner une validation interne. Ce découpage séparerait les
  usagers d'un même accident, alors que tout le projet les garde groupés. Le nombre
  d'itérations est donc placé dans la grille et choisi sur 2023, comme les autres réglages.

In [ ]:
PROTOCOLE = {
    "version": "1.0",
    "date_ecriture_utc": datetime.now(timezone.utc).isoformat(),
    "objet": "comparaison d'un modèle d'arbres boostés à la régression logistique du 06",
    "cible": CIBLE,
    "unite": "un usager à gravité connue",
    "apprentissage": APPRENTISSAGE,
    "validation": VALIDATION,
    "test": TEST,
    "variables_modele": list(X_app.columns),
    "representation": (
        "mêmes 11 variables et mêmes tranches d'âge que le notebook 06 ; modalités "
        "apprises sur 2019-2022 et déclarées en catégories pandas ; une modalité jamais "
        "vue devient une valeur manquante, traitée comme telle, et elle est comptée"
    ),
    "modele": "HistGradientBoostingClassifier",
    "parametres_fixes": {
        "min_samples_leaf": 50, "l2_regularization": 0.0, "early_stopping": False,
        "categorical_features": "from_dtype", "random_state": 20261017,
    },
    "justification_early_stopping": (
        "désactivé : son découpage interne est aléatoire et séparerait les usagers d'un "
        "même accident ; le nombre d'itérations est choisi sur 2023 comme le reste"
    ),
    "grille": {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31],
               "max_iter": [200, 400]},
    "selection": (
        "log-loss 2023 minimale ; égalité départagée par le modèle le plus simple "
        "(max_leaf_nodes, puis max_iter, puis learning_rate les plus petits)"
    ),
    "comparaison_finale": (
        "le modèle d'arbres n'est retenu que si sa log-loss de validation est "
        "strictement inférieure à celle de la régression logistique"
    ),
    "limite_declaree": (
        "les scores de 2024 de la régression logistique ont déjà été consultés le "
        "17 octobre 2026 ; 2024 n'est plus une année vierge pour cette comparaison. "
        "Les réglages sont choisis sur 2023 seule"
    ),
    "sha256_donnees": sha_donnees,
    "environnement": {
        "python": sys.version.split()[0], "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

with open(os.path.join(TABLES, "08_protocole_arbres.json"), "w") as f:
    json.dump(PROTOCOLE, f, ensure_ascii=False, indent=2)
print("Protocole enregistré avant tout calcul de score.")

In [ ]:
def en_categories(X, categories):
    '''Déclarer les modalités apprises ; une modalité inconnue devient manquante.'''
    sortie = pd.DataFrame(index=X.index)
    inconnues = {}
    for colonne in X.columns:
        serie = X[colonne].astype(object)
        inconnues[colonne] = int((~serie.isin(set(categories[colonne]))).sum())
        sortie[colonne] = pd.Categorical(serie, categories=categories[colonne])
    return sortie, inconnues


categories = {c: sorted(X_app[c].astype(object).unique()) for c in X_app.columns}
C_app, _ = en_categories(X_app, categories)
C_val, inconnues_val = en_categories(X_val, categories)
C_tst, inconnues_tst = en_categories(X_tst, categories)

print("modalités jamais vues en validation :", {k: v for k, v in inconnues_val.items() if v} or "aucune")
print("modalités jamais vues au test       :", {k: v for k, v in inconnues_tst.items() if v} or "aucune")

## 4. Sélection sur 2023

Huit configurations, un seul critère : la log-loss de 2023.

In [ ]:
fixes = PROTOCOLE["parametres_fixes"]
grille = PROTOCOLE["grille"]

lignes = []
for lr, feuilles, iterations in itertools.product(
    grille["learning_rate"], grille["max_leaf_nodes"], grille["max_iter"]
):
    debut = time.time()
    modele = HistGradientBoostingClassifier(
        learning_rate=lr, max_leaf_nodes=feuilles, max_iter=iterations,
        min_samples_leaf=fixes["min_samples_leaf"],
        l2_regularization=fixes["l2_regularization"],
        early_stopping=False, categorical_features="from_dtype",
        random_state=fixes["random_state"],
    )
    modele.fit(C_app, y_app)
    ligne = scores(y_val, modele.predict_proba(C_val)[:, 1])
    ligne.update({"learning_rate": lr, "max_leaf_nodes": feuilles,
                  "max_iter": iterations, "secondes": round(time.time() - debut, 2)})
    lignes.append(ligne)

selection = pd.DataFrame(lignes)[
    ["learning_rate", "max_leaf_nodes", "max_iter", "log_loss", "brier",
     "average_precision", "roc_auc", "secondes"]
].sort_values(["log_loss", "max_leaf_nodes", "max_iter", "learning_rate"]).reset_index(drop=True)
selection.to_csv(os.path.join(TABLES, "08_selection_arbres.csv"), index=False)

retenu = selection.iloc[0]
print(selection.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print(f"\nRetenu : learning_rate={retenu.learning_rate}, "
      f"max_leaf_nodes={int(retenu.max_leaf_nodes)}, max_iter={int(retenu.max_iter)}")

**Lecture.** Les huit configurations tiennent dans un intervalle de log-loss de moins de
0,002. Comme pour le choix de `C` dans le notebook 06, le réglage n'est pas déterminant :
c'est la famille de modèle qui fait la différence, pas son accordage. Il faut le dire,
plutôt que de présenter la sélection comme un travail d'optimisation décisif.

## 5. Évaluation sur 2024

Configuration arrêtée, même apprentissage 2019-2022, aucun réentraînement sur 2023.

In [ ]:
arbres = HistGradientBoostingClassifier(
    learning_rate=float(retenu.learning_rate),
    max_leaf_nodes=int(retenu.max_leaf_nodes),
    max_iter=int(retenu.max_iter),
    min_samples_leaf=fixes["min_samples_leaf"],
    l2_regularization=fixes["l2_regularization"],
    early_stopping=False, categorical_features="from_dtype",
    random_state=fixes["random_state"],
)
arbres.fit(C_app, y_app)
p_arb_val = arbres.predict_proba(C_val)[:, 1]
p_arb_tst = arbres.predict_proba(C_tst)[:, 1]

comparaison = []
for annee, y, p_log, p_arb in [(VALIDATION, y_val, p_log_val, p_arb_val),
                               (TEST, y_tst, p_log_tst, p_arb_tst)]:
    reference = np.full_like(y, probabilite_reference, dtype=float)
    for nom, p in [("Référence constante", reference),
                   ("Régression logistique", p_log),
                   ("Arbres boostés", p_arb)]:
        comparaison.append({"annee": annee, "modele": nom, **scores(y, p)})

comparaison = pd.DataFrame(comparaison)
comparaison.to_csv(os.path.join(TABLES, "08_comparaison_modeles.csv"), index=False)
print(comparaison.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

assert (comparaison.query("annee == @VALIDATION and modele == 'Arbres boostés'")["log_loss"].iloc[0]
        < comparaison.query("annee == @VALIDATION and modele == 'Régression logistique'")["log_loss"].iloc[0]), \
    "Règle du protocole : conserver la logistique si les arbres ne font pas strictement mieux."

**Lecture.** Le modèle d'arbres améliore les quatre scores, sur 2023 comme sur 2024.
L'écart entre les deux années est négligeable pour les deux modèles : ce qui est appris
sur 2019-2022 reste valable deux ans plus tard.

Le gain le plus net porte sur l'*average precision* : le classement des usagers les plus
exposés s'améliore beaucoup plus que la log-loss ne le laisse voir. C'est ce que confirme
la section suivante.

## 6. Pourquoi les arbres font mieux

La régression logistique est additive sur l'échelle des log-cotes : chaque variable
déplace le risque d'une quantité fixe, indépendamment des autres. Si deux caractéristiques
se renforcent, le modèle ne peut pas l'exprimer.

On le vérifie de deux façons : par la calibration, puis par les profils croisés.

In [ ]:
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150, "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.autolayout": True})
COULEURS = {"Référence constante": "#8c8c8c", "Régression logistique": "#1f77b4",
            "Arbres boostés": "#d62728"}

bornes = np.linspace(0, 1, 11)
fig, ax = plt.subplots(figsize=(7, 5.2))
ax.plot([0, 1], [0, 1], "--", color="black", lw=1, label="Calibration parfaite")
for nom, p in [("Régression logistique", p_log_tst), ("Arbres boostés", p_arb_tst)]:
    idx = np.clip(np.digitize(p, bornes) - 1, 0, 9)
    xs, ys = [], []
    for b in range(10):
        m = idx == b
        if m.sum() >= 50:
            xs.append(p[m].mean()); ys.append(y_tst[m].mean())
    ax.plot(xs, ys, "o-", color=COULEURS[nom], label=nom, markersize=5)
ax.axhline(y_tst.mean(), color=COULEURS["Référence constante"], ls=":", lw=1.2,
           label=f"Taux observé ({100 * y_tst.mean():.1f} %)")
ax.set_xlabel("Probabilité moyenne estimée")
ax.set_ylabel("Proportion observée de décès ou hospitalisation")
ax.set_title("Calibration sur 2024\nDix intervalles fixes, intervalles de moins de 50 usagers écartés")
ax.legend(frameon=False, loc="upper left")
fig.savefig(os.path.join(FIGURES, "08_01_calibration_comparaison_test.png"))
plt.show()

**Lecture.** En dessous de 0,45 les deux modèles se tiennent sur la diagonale. Au-delà,
la régression logistique décroche : elle n'attribue presque jamais de probabilité
supérieure à 0,85, et dans l'intervalle 0,6-0,8 elle sous-estime nettement le risque
réellement observé. Les arbres suivent la diagonale jusqu'au bout.

Autrement dit, le modèle linéaire **ne sait pas désigner les cas les plus graves**.

In [ ]:
LIB_CATU = {"1": "Conducteur", "2": "Passager", "3": "Piéton"}
profils = X_tst.copy()
profils["categorie"] = profils["catu"].map(LIB_CATU).fillna(profils["catu"])
profils = profils.assign(observe=y_tst, logistique=p_log_tst, arbres=p_arb_tst)

croises = (profils.groupby(["categorie", "tranche_age"], observed=True)
           .agg(usagers=("observe", "size"),
                observe_pct=("observe", lambda s: 100 * s.mean()),
                logistique_pct=("logistique", lambda s: 100 * s.mean()),
                arbres_pct=("arbres", lambda s: 100 * s.mean()))
           .reset_index())
croises = croises[croises["usagers"] >= 300].copy()
croises["ecart_logistique"] = croises["logistique_pct"] - croises["observe_pct"]
croises["ecart_arbres"] = croises["arbres_pct"] - croises["observe_pct"]
croises = croises.sort_values("observe_pct", ascending=False)
croises.to_csv(os.path.join(TABLES, "08_profils_croises_test.csv"), index=False)

print(croises.head(8).to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print("\nÉcart absolu moyen à l'observation, en points de pourcentage")
print(f"  régression logistique : {croises['ecart_logistique'].abs().mean():.2f}")
print(f"  arbres boostés        : {croises['ecart_arbres'].abs().mean():.2f}")

**Lecture.** Sur les profils croisés « catégorie d'usager × tranche d'âge », l'écart
absolu moyen à la réalité est environ trois fois plus faible pour les arbres.

Le cas le plus parlant est celui des **conducteurs de moins de 18 ans** : ce sont pour
l'essentiel des conducteurs de deux-roues. La proportion réelle de décès ou
d'hospitalisation y est d'environ 32 %, alors que la régression logistique en annonce
près de 26 %. Elle additionne un effet « 0-17 ans » modéré et un effet « conducteur »
neutre, sans pouvoir exprimer que la combinaison des deux est bien plus dangereuse que
la somme de ses parties. Les arbres retrouvent le bon niveau.

## 7. L'arbitrage précision-rappel

Le seuil de 0,5 des notebooks 06 et 07 est illustratif. Il ne répond pas à la question
utile : **que coûte un rappel plus élevé ?**

Choisir le seuil directement sur 2024 reviendrait à régler quelque chose sur l'année de
test. On fixe donc chaque seuil sur 2023, pour y atteindre un rappel visé, puis on
l'applique sans retouche à 2024.

In [ ]:
RAPPELS_CIBLES = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]


def seuil_pour_rappel(y, p, rappel_cible):
    '''Plus grand seuil atteignant au moins le rappel visé.'''
    ordre = np.argsort(-p)
    cumul = np.cumsum(y[ordre])
    besoin = int(np.ceil(rappel_cible * y.sum()))
    k = int(np.searchsorted(cumul, besoin)) + 1
    return float(p[ordre][k - 1])


def mesures(y, p, seuil):
    alerte = p >= seuil
    vp = int((alerte & (y == 1)).sum()); fp = int((alerte & (y == 0)).sum())
    fn = int((~alerte & (y == 1)).sum())
    return {"alertes": vp + fp, "vrais_positifs": vp, "fausses_alertes": fp,
            "cas_manques": fn,
            "precision": vp / (vp + fp) if vp + fp else np.nan,
            "rappel": vp / (vp + fn) if vp + fn else np.nan,
            "fausses_alertes_par_cas_detecte": fp / vp if vp else np.nan,
            "part_population_alertee_pct": 100 * (vp + fp) / len(y)}


lignes = []
for nom, p_val, p_tst in [("Régression logistique", p_log_val, p_log_tst),
                          ("Arbres boostés", p_arb_val, p_arb_tst)]:
    lignes.append({"modele": nom, "regle_du_seuil": "seuil 0,5 (notebooks 06-07)",
                   "rappel_vise_sur_2023": np.nan, "seuil": 0.5,
                   **mesures(y_tst, p_tst, 0.5)})
    for cible in RAPPELS_CIBLES:
        seuil = seuil_pour_rappel(y_val, p_val, cible)
        lignes.append({"modele": nom,
                       "regle_du_seuil": "seuil fixé sur 2023, appliqué à 2024",
                       "rappel_vise_sur_2023": cible, "seuil": seuil,
                       **mesures(y_tst, p_tst, seuil)})

seuils = pd.DataFrame(lignes)
seuils.to_csv(os.path.join(TABLES, "08_seuils_rappel_cible.csv"), index=False)
pd.set_option("display.width", 200)
print(seuils[["modele", "rappel_vise_sur_2023", "seuil", "rappel", "precision",
              "fausses_alertes", "cas_manques", "fausses_alertes_par_cas_detecte",
              "part_population_alertee_pct"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.2))
for nom, p in [("Régression logistique", p_log_tst), ("Arbres boostés", p_arb_tst)]:
    precision, rappel, _ = precision_recall_curve(y_tst, p)
    ax.plot(rappel, precision, color=COULEURS[nom], lw=1.8, label=nom)
    points = seuils[(seuils["modele"] == nom) & seuils["rappel_vise_sur_2023"].notna()]
    ax.plot(points["rappel"], points["precision"], "o", color=COULEURS[nom],
            markersize=6, markeredgecolor="white", markeredgewidth=1)
ax.axhline(y_tst.mean(), color=COULEURS["Référence constante"], ls=":", lw=1.2,
           label=f"Hasard ({100 * y_tst.mean():.1f} %)")
ax.set_xlabel("Rappel : part des cas graves détectés")
ax.set_ylabel("Précision : part d'alertes justes")
ax.set_title("Arbitrage précision-rappel sur 2024\nLes points marquent les seuils fixés sur 2023")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(frameon=False)
fig.savefig(os.path.join(FIGURES, "08_02_precision_rappel_comparaison_test.png"))
plt.show()

**Lecture.** Deux enseignements.

D'abord, **les seuils se transportent**. Un seuil réglé sur 2023 pour un rappel de 50 %
donne un rappel de 49,7 % à 50,2 % sur 2024 selon le modèle. La stabilité temporelle ne
concerne donc pas seulement les scores agrégés : elle vaut aussi pour une règle de
décision concrète.

Ensuite, **l'arbitrage est explicite**. Pour détecter la moitié des cas graves de 2024,
il faut signaler environ un usager sur cinq, et un peu plus d'une alerte sur deux sera
fausse. Dire « le rappel est de 18 % » sans montrer cette courbe donne une image
faussement négative : le modèle n'échoue pas, c'est le seuil de 0,5 qui est arbitraire.

Le modèle d'arbres domine la régression logistique à tous les niveaux de rappel.

## 8. Les usagers sans âge

Le notebook 07 relève que les usagers dont l'âge est absent ne présentent presque jamais
de cas grave. On examine ici d'où vient ce groupe et ce qu'il apporte aux scores.

In [ ]:
connus = connus.assign(age_absent=connus["age_approx"].isna())

lignes = []
for annee, bloc in connus.groupby("annee_source"):
    absent = bloc[bloc["age_absent"]]
    present = bloc[~bloc["age_absent"]]
    lignes.append({
        "annee": int(annee), "usagers": len(bloc), "age_absent": len(absent),
        "part_age_absent_pct": 100 * len(absent) / len(bloc),
        "dont_age_juge_invraisemblable": int(bloc["age_suspect"].fillna(False).astype(bool).sum()),
        "positifs_si_age_absent": int(absent[CIBLE].sum()),
        "taux_si_age_absent_pct": 100 * absent[CIBLE].mean() if len(absent) else np.nan,
        "taux_si_age_present_pct": 100 * present[CIBLE].mean(),
    })
annuel = pd.DataFrame(lignes)
annuel.to_csv(os.path.join(TABLES, "08_ages_manquants_par_annee.csv"), index=False)
print(annuel.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
absents = connus[connus["age_absent"]]
profil = []
for champ, libelle in [("catu", "catégorie d'usager"), ("sexe", "sexe"),
                       ("grav", "gravité enregistrée")]:
    for modalite, n in absents[champ].astype("string").fillna("Non renseigné").value_counts().items():
        profil.append({"champ": libelle, "modalite": modalite, "usagers": int(n),
                       "part_pct": 100 * n / len(absents)})
profil = pd.DataFrame(profil)
profil.to_csv(os.path.join(TABLES, "08_ages_manquants_profil.csv"), index=False)
print(f"Profil des {len(absents)} usagers sans âge, gravité connue, 2019-2024\n")
print(profil.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.6))
ax1.bar(annuel["annee"], annuel["part_age_absent_pct"], color="#4c72b0", width=0.65)
for _, r in annuel.iterrows():
    ax1.text(r["annee"], r["part_age_absent_pct"] + 0.05, f"{r['age_absent']:.0f}",
             ha="center", fontsize=8)
ax1.set_xlabel("Année source"); ax1.set_ylabel("Part des usagers sans âge (%)")
ax1.set_title("L'absence d'âge apparaît en 2021"); ax1.set_ylim(0, 2.8)

largeur, x = 0.38, np.arange(len(annuel))
ax2.bar(x - largeur / 2, annuel["taux_si_age_present_pct"], largeur,
        label="Âge renseigné", color="#4c72b0")
ax2.bar(x + largeur / 2, annuel["taux_si_age_absent_pct"], largeur,
        label="Âge absent", color="#c44e52")
for i, r in annuel.iterrows():
    if r["age_absent"] > 0:
        ax2.text(i + largeur / 2, r["taux_si_age_absent_pct"] + 0.4,
                 f"n={r['age_absent']:.0f}", ha="center", fontsize=7, color="#c44e52")
ax2.set_xticks(x); ax2.set_xticklabels(annuel["annee"].astype(int))
ax2.set_xlabel("Année source"); ax2.set_ylabel("Décès ou hospitalisation (%)")
ax2.set_title("Un groupe presque jamais gravement blessé"); ax2.set_ylim(0, 21)
ax2.legend(frameon=False, loc="upper center", ncol=2)
fig.suptitle("Les usagers sans âge : une pratique d'enregistrement, pas un profil de risque",
             fontsize=11)
fig.savefig(os.path.join(FIGURES, "08_03_ages_manquants.png"))
plt.show()

**Ce que montrent ces deux tableaux.**

L'absence d'âge **apparaît en 2021**. En 2019 et 2020, les seuls âges manquants sont les
162 et 10 valeurs que le notebook 03 a lui-même rendues manquantes parce qu'elles étaient
invraisemblables. À partir de 2021, environ 2 % des usagers arrivent sans âge dans la
source, soit entre 2 300 et 2 900 personnes par an, et aucun ne provient de notre
nettoyage. C'est donc un changement de pratique d'enregistrement, pas un effet de notre
préparation.

Ces usagers forment un groupe très homogène : **99,5 % sont des conducteurs**, **94 %
n'ont pas non plus de sexe renseigné**, et **99,4 % sont enregistrés comme indemnes**.
L'interprétation la plus plausible est administrative : une ligne d'usager créée pour
représenter le véhicule, sans que l'identité du conducteur soit relevée.

Le taux de décès ou d'hospitalisation y est de 0 % à 0,04 % selon l'année, contre environ
18 % ailleurs. Les modèles l'apprennent et attribuent à ce groupe une probabilité de
l'ordre de 0,002.

In [ ]:
masque_present = (~test["age_approx"].isna()).to_numpy()
jeux = {"Référence constante": np.full_like(y_tst, probabilite_reference, dtype=float),
        "Régression logistique": p_log_tst, "Arbres boostés": p_arb_tst}

impact = []
for nom, p in jeux.items():
    for libelle, m in [("tous les usagers de 2024", np.ones(len(y_tst), dtype=bool)),
                       ("âge renseigné seulement", masque_present)]:
        impact.append({"modele": nom, "population": libelle, "usagers": int(m.sum()),
                       **scores(y_tst[m], p[m])})
impact = pd.DataFrame(impact)
impact.to_csv(os.path.join(TABLES, "08_ages_manquants_impact_scores.csv"), index=False)
print(impact.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

**Ce que ce groupe apporte aux scores.**

Retirer ces 2 419 usagers de l'évaluation de 2024 dégrade tous les scores : la ROC-AUC
de la régression logistique passe de 0,770 à 0,764, celle des arbres de 0,789 à 0,784.
Environ **0,6 point de ROC-AUC vient donc d'un artefact d'enregistrement**, et non d'une
information sur le risque routier.

Un détail important : l'*average precision*, elle, ne bouge pratiquement pas. Ces usagers
reçoivent des probabilités très basses et sont effectivement négatifs ; ils se rangent
tout en bas du classement sans peser sur la courbe précision-rappel. Les scores qui
dépendent de l'ensemble de la distribution (log-loss, Brier, ROC-AUC) sont flattés, celui
qui se concentre sur le haut du classement ne l'est pas.

**Ce n'est pas une fuite de cible** au sens strict : l'information est disponible au même
moment que les autres variables et n'est pas dérivée de la gravité. Mais c'est un
indicateur du processus de saisie plutôt que du risque. Les conclusions du projet
tiennent sans lui, ce que confirme l'évaluation restreinte ci-dessus.

## 9. Conclusion

**Sur le modèle.** Les arbres boostés améliorent les quatre scores par rapport à la
régression logistique, sur 2023 comme sur 2024, avec la même représentation et le même
protocole. Le gain vient de la capacité à représenter des combinaisons de caractéristiques
que l'additivité des log-cotes ne peut pas exprimer. La régression logistique reste le
modèle le plus lisible : ses contrastes du notebook 06 s'interprètent directement, ce que
les arbres ne permettent pas.

**Sur les seuils.** Le rappel de 18 % au seuil 0,5 n'est pas une limite du modèle mais une
conséquence du seuil choisi. La table des seuils rend l'arbitrage explicite et montre
qu'une règle fixée sur 2023 se comporte comme prévu sur 2024.

**Sur les usagers sans âge.** Le phénomène est daté, décrit et quantifié. Il gonfle
légèrement les scores globaux sans remettre en cause les conclusions.

**Ce que cela ne démontre toujours pas.** Aucune relation causale. Aucune utilité
opérationnelle : ni aide au secours, ni prévention individuelle, ni décision
d'aménagement. Le périmètre reste la gravité conditionnelle à l'implication dans un
accident corporel enregistré, sans données d'exposition.

**Limite de protocole à redire telle quelle.** 2024 avait déjà été consultée avant cette
comparaison. Les réglages du modèle d'arbres ont été choisis sur 2023 seule, mais une
année réellement vierge serait nécessaire pour trancher définitivement entre les deux
modèles. La stabilité observée entre 2023 et 2024 rend le résultat crédible ; elle ne
remplace pas un test non consulté.